# Your first evaluation

[Read the guide](https://docs.futureagi.com/docs/cookbook/quickstart/first-eval). Use a Python 3.12 notebook with Bash and venv support. Start with local metrics; you only need keys for the hosted metrics and custom judge.

## Install

In [ ]:
import os
import tempfile

# Use a fresh directory rather than overwriting an existing example.
workspace = tempfile.mkdtemp(prefix="ridgeline-")
os.chdir(workspace)
os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
os.environ["PIP_QUIET"] = "1"

In [ ]:
%%bash
set -euo pipefail
python3.12 -m venv .venv
source .venv/bin/activate
python -m pip install agent-learning-kit==0.3.7

## Run a local metric (no API key)

In [ ]:
%%script .venv/bin/python
from fi.evals import evaluate

result = evaluate("contains", output="Your order has shipped!", keyword="shipped")

print(result.score)   # 1.0
print(result.passed)  # True
print(result.reason)  # "Keyword 'shipped' found"

In [ ]:
%%script .venv/bin/python
from fi.evals import evaluate

assert evaluate("equals", output="Paris", expected_output="Paris").passed          # True
assert evaluate("is_json", output='{"status": "ok"}').passed                       # True
assert evaluate("length_less_than", output="Short reply.", config={"max_length": 100}).passed   # True

result = evaluate("levenshtein_similarity", output="colour", expected_output="color")
print(result.score)  # similarity score between 0 and 1

## Detect contradictions with local NLI

The first model load can download weights. If the model cannot load, the word-overlap fallback can pass both examples.

In [ ]:
%%bash
set -euo pipefail
source .venv/bin/activate
# The kit runs NLI on CPU. On Linux, install the CPU build first.
if [ "$(uname -s)" = Linux ]; then
  python -m pip install "torch>=2.10,<3" --index-url https://download.pytorch.org/whl/cpu
fi
python -m pip install "agent-learning-kit[nli]==0.3.7" sentencepiece

In [ ]:
%%script .venv/bin/python
from fi.evals import evaluate

# Supported response
result = evaluate(
    "contradiction_detection",
    output="The Eiffel Tower is located in Paris, France.",
    context="The Eiffel Tower is a wrought-iron lattice tower located in Paris.",
)
print(f"Score: {result.score:.2f}")
print(f"Passed: {result.passed}")

# Contradictory response
result = evaluate(
    "contradiction_detection",
    output="The Eiffel Tower is located in London, England.",
    context="The Eiffel Tower is a wrought-iron lattice tower located in Paris.",
)
print(f"Score: {result.score:.2f}")
print(f"Passed: {result.passed}")
print(f"Why: {result.reason}")

## Score with Future AGI Turing models

Use environment variables or enter the keys at the hidden prompts.

In [ ]:
import os
from getpass import getpass

def require_secret(name):
    if os.environ.get(name):
        return
    value = getpass(f"{name} (hidden): ")
    if not value.strip():
        raise ValueError(f"{name} is required for this step")
    os.environ[name] = value.strip()

require_secret("FI_API_KEY")
require_secret("FI_SECRET_KEY")

In [ ]:
%%writefile toxicity.py
from fi.evals import evaluate

ids = []
for reply in ["You're amazing, keep it up!", "I hate you and everything you stand for."]:
    result = evaluate("toxicity", output=reply, model="turing_small")
    if result.score is None:
        raise RuntimeError(result.error or result.reason or "No toxicity score returned")
    print(f"Score: {result.score}, passed: {result.passed}. {result.reason}")
    ids.append(result.metadata["eval_id"])
print(*ids)

In [ ]:
%%bash
set -euo pipefail
source .venv/bin/activate
python toxicity.py

## Run multiple metrics at once

In [ ]:
%%writefile batch.py
from fi.evals import evaluate

results = evaluate(
    ["toxicity", "groundedness"],
    output="The Eiffel Tower is located in Paris, France.",
    context="The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris.",
    input="Where is the Eiffel Tower?",
    model="turing_small",
)

for result in results:
    if result.score is None:
        raise RuntimeError(result.error or result.reason or "Missing score")
    status = "PASS" if result.passed else "FAIL"
    print(f"{result.eval_name:<20} score={result.score}  {status}")
    print(f"  Reason: {result.reason}\n")
print(*(r.metadata["eval_id"] for r in results))

In [ ]:
%%bash
set -euo pipefail
source .venv/bin/activate
python batch.py

## Write your own evaluation criteria (LLM-as-Judge)

In [ ]:
require_secret("OPENAI_API_KEY")

In [ ]:
%%script .venv/bin/python
from fi.evals import evaluate

result = evaluate(
    prompt="""You are evaluating a customer support response.

    Score 1.0 if the response:
    - Acknowledges the customer's issue clearly
    - Offers a concrete next step or resolution
    - Stays professional and empathetic

    Score 0.5 if it's polite but vague (no clear next step).
    Score 0.0 if it's dismissive, rude, or unhelpful.""",
    output="I understand your frustration with the delayed shipment. I've escalated this to our logistics team and you'll receive a status update within 2 hours.",
    input="My order is 3 weeks late and nobody is responding to my emails.",
    engine="llm",
    model="gpt-4o-mini",
)

if result.score is None:
    raise RuntimeError(result.error or result.reason or "No judge score returned")
print(f"Score: {result.score}")
print(f"Why: {result.reason}")

## Run evaluations on a dataset from the dashboard

For scores alongside rows, open a dataset with an `output` column, select **Evaluate**, add `toxicity`, and map its input to `output`. See [Run batch evaluations on a CSV](https://docs.futureagi.com/docs/cookbook/quickstart/batch-eval).